# Week 3 — Data Cleaning and Preprocessing

**Machine Learning in Finance · NKUA MPhil Economics · 2026–2027**

This notebook follows the lecture presentation. It uses the complete UCI credit-card default table stored in the course repository. The source contains no blank cells, so the missingness examples are controlled in-memory demonstrations.

**Learning route:** audit → clean → represent → split → transform → verify.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_URL = (
    "https://raw.githubusercontent.com/fkottas/Teaching-Courses/main/"
    "Machine%20Learning%20in%20Finance/2026-2027/lectures/lecture-00-01/"
    "data/credit/default_of_credit_card_clients.csv.gz"
)

credit = pd.read_csv(DATA_URL, compression="gzip")
credit = credit.rename(columns={"default payment next month": "default_next_month"})
print("shape:", credit.shape)
print(credit.head(3).to_string(index=False))

shape: (30000, 25)
 ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default_next_month
  1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1
  2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1
  3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                   0


## 1. Audit the table before changing it

The audit describes what is stored. It does not yet decide which values are valid predictors. We inspect data types, missingness, uniqueness, duplicates, target balance, and coded categories.

In [2]:
audit = pd.DataFrame({
    "dtype": credit.dtypes.astype(str),
    "missing": credit.isna().sum(),
    "unique": credit.nunique(),
})
print(audit.head(10).to_string())
print("\nexact duplicate rows:", int(credit.duplicated().sum()))
print("unique IDs:", int(credit["ID"].nunique()))

           dtype  missing  unique
ID         int64        0   30000
LIMIT_BAL  int64        0      81
SEX        int64        0       2
EDUCATION  int64        0       7
MARRIAGE   int64        0       4
AGE        int64        0      56
PAY_0      int64        0      11
PAY_2      int64        0      11
PAY_3      int64        0      11
PAY_4      int64        0      11

exact duplicate rows: 0
unique IDs: 30000


In [3]:
print("target counts:")
print(credit["default_next_month"].value_counts().sort_index())
print("\nEDUCATION codes:")
print(credit["EDUCATION"].value_counts().sort_index())
print("\nSEX codes:")
print(credit["SEX"].value_counts().sort_index())

target counts:
default_next_month
0    23364
1     6636
Name: count, dtype: int64

EDUCATION codes:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

SEX codes:
SEX
1    11888
2    18112
Name: count, dtype: int64


## 2. Missing values: demonstrate the rule without changing the source

The UCI course copy has zero blank cells. To practise a missingness policy, create a small copy, introduce known missing cells, preserve a missingness indicator, and then impute the numeric value using the demonstration sample. In a real pipeline, the imputation statistic must be learned from training data only.

In [4]:
demo = credit[["LIMIT_BAL", "AGE", "EDUCATION", "MARRIAGE"]].head(8).copy()
demo.loc[[1, 4], "AGE"] = np.nan
demo.loc[[2], "EDUCATION"] = np.nan
missing_rate = demo.isna().mean()
print(missing_rate.to_string())
demo["AGE_was_missing"] = demo["AGE"].isna().astype(int)
demo["AGE"] = demo["AGE"].fillna(demo["AGE"].median())
print("\nAge after imputation and missingness flag:")
print(demo[["AGE", "AGE_was_missing"]].to_string(index=False))

LIMIT_BAL    0.000
AGE          0.250
EDUCATION    0.125
MARRIAGE     0.000

Age after imputation and missingness flag:
 AGE  AGE_was_missing
24.0                0
31.5                1
34.0                0
37.0                0
31.5                1
37.0                0
29.0                0
23.0                0


## 3. Duplicates and identity checks

An exact duplicate is a repeated row across all stored columns. A repeated identifier is a separate check: it may indicate a duplicate record, a repeated event, or a data-model problem. We demonstrate both checks on an in-memory example.

In [5]:
duplicate_demo = pd.concat([credit.head(5), credit.head(1)], ignore_index=True)
print("rows:", len(duplicate_demo))
print("exact duplicates:", int(duplicate_demo.duplicated().sum()))
print("duplicated IDs:", int(duplicate_demo["ID"].duplicated().sum()))

rows: 6
exact duplicates: 1
duplicated IDs: 1


## 4. Outliers: flag first, decide later

A large credit limit or payment amount may be valid. The code below uses the 1st and 99th percentiles for a transparent teaching flag. The flag does not delete observations and is not a claim that the rows are errors.

In [6]:
lower, upper = credit["LIMIT_BAL"].quantile([0.01, 0.99])
limit_flag = ~credit["LIMIT_BAL"].between(lower, upper)
print("lower threshold:", lower)
print("upper threshold:", upper)
print("within 1%-99%:", int((~limit_flag).sum()))
print("outside 1%-99%:", int(limit_flag.sum()))

lower threshold: 10000.0
upper threshold: 500000.0
within 1%-99%: 29794
outside 1%-99%: 206


In [7]:
outlier_summary = pd.DataFrame({
    "variable": ["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1"],
    "q1": [50000, 28, 3558.75, 1000],
    "median": [140000, 34, 22381.5, 2100],
    "q3": [240000, 41, 67091, 5006],
})
print(outlier_summary.to_string(index=False))

 variable       q1   median     q3
LIMIT_BAL 50000.00 140000.0 240000
      AGE    28.00     34.0     41
BILL_AMT1  3558.75  22381.5  67091
 PAY_AMT1  1000.00   2100.0   5006


## 5. Separate variable types before encoding

The teaching block uses five numeric predictors and three categorical predictors. The identifier remains available for traceability but is excluded from `X`; the target is kept in `y`.

In [8]:
num_cols = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
X = credit[num_cols + cat_cols]
y = credit["default_next_month"]
print("numeric:", num_cols)
print("categorical:", cat_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

numeric: ['LIMIT_BAL', 'AGE', 'PAY_0', 'BILL_AMT1', 'PAY_AMT1']
categorical: ['SEX', 'EDUCATION', 'MARRIAGE']
X shape: (30000, 8)
y shape: (30000,)


In [9]:
cats = credit[["SEX", "EDUCATION", "MARRIAGE"]].head()
encoded = pd.get_dummies(cats.astype("string"), drop_first=False)
print(encoded.shape)
print(encoded.columns.tolist())

(5, 5)
['SEX_1', 'SEX_2', 'EDUCATION_2', 'MARRIAGE_1', 'MARRIAGE_2']


## 6. Split first, then fit transformations

The test rows are held out before any imputer, scaler, or encoder learns values. Stratification keeps the binary target proportion similar across the two partitions.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train default rate:", round(y_train.mean(), 4))
print("y_test default rate:", round(y_test.mean(), 4))

X_train: (24000, 8)
X_test: (6000, 8)
y_train default rate: 0.2212
y_test default rate: 0.2212


In [11]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train[["LIMIT_BAL", "AGE"]])
test_scaled = scaler.transform(X_test[["LIMIT_BAL", "AGE"]])
print(train_scaled.shape, test_scaled.shape)
print("training means:", train_scaled.mean(axis=0).round(4))
print("training std:", train_scaled.std(axis=0).round(4))

(24000, 2) (6000, 2)
training means: [ 0. -0.]
training std: [1. 1.]


## 7. Build one leakage-safe `ColumnTransformer`

The numeric pipeline imputes with the training median and then scales. The categorical pipeline imputes with the training mode and then one-hot encodes. `handle_unknown="ignore"` keeps a new category from breaking a later prediction.

In [12]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_encoder()),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("X_train_ready:", X_train_ready.shape)
print("X_test_ready:", X_test_ready.shape)

X_train_ready: (24000, 18)
X_test_ready: (6000, 18)


In [13]:
feature_names = preprocessor.get_feature_names_out()
print("number of output features:", len(feature_names))
print(feature_names.tolist())

number of output features: 18
['num__LIMIT_BAL', 'num__AGE', 'num__PAY_0', 'num__BILL_AMT1', 'num__PAY_AMT1', 'cat__SEX_1', 'cat__SEX_2', 'cat__EDUCATION_0', 'cat__EDUCATION_1', 'cat__EDUCATION_2', 'cat__EDUCATION_3', 'cat__EDUCATION_4', 'cat__EDUCATION_5', 'cat__EDUCATION_6', 'cat__MARRIAGE_0', 'cat__MARRIAGE_1', 'cat__MARRIAGE_2', 'cat__MARRIAGE_3']


## Submission checklist

Before submitting, confirm that the notebook runs from top to bottom in a fresh Colab runtime, the source and variables are documented, controlled examples are clearly labelled, outlier treatment is not automatic, and every transformation is fitted only on the training partition.

## Worked cases: connect each rule to a financial decision

The following small cases reproduce the additional examples in the presentation. Run each one, inspect the output, and write one sentence explaining the decision.

In [14]:
case_a = pd.DataFrame({
    "borrower": ["A", "B", "C", "D", "E"],
    "income": [42000.0, np.nan, 58000.0, np.nan, 72000.0],
})
case_a["income_was_missing"] = case_a["income"].isna().astype(int)
median_income = case_a["income"].median()
case_a["income"] = case_a["income"].fillna(median_income)
print("median income:", median_income)
print(case_a.to_string(index=False))

median income: 58000.0
borrower  income  income_was_missing
       A 42000.0                   0
       B 58000.0                   1
       C 58000.0                   0
       D 58000.0                   1
       E 72000.0                   0


### Case A — missing income

The imputed value makes the column usable for a numeric transformer. The indicator preserves the fact that the source did not report income.

In [15]:
transactions = pd.DataFrame({
    "transaction_id": [101, 102, 102, 103],
    "amount": [120, 75, 75, 240],
    "merchant": ["A", "B", "B", "C"],
})
exact_duplicate = transactions.duplicated()
repeated_id = transactions["transaction_id"].duplicated()
print("exact duplicate rows:", int(exact_duplicate.sum()))
print("repeated transaction IDs:", int(repeated_id.sum()))
print(transactions.loc[exact_duplicate].to_string(index=False))

exact duplicate rows: 1
repeated transaction IDs: 1
 transaction_id  amount merchant
            102      75        B


### Case B — duplicate transaction

The row-level check and the identifier check answer different questions. Review the event definition before deleting anything.

In [16]:
payments = pd.DataFrame({"amount": [800, 950, 1100, 1300, 1500, 1600, 180000]})
q1 = payments["amount"].quantile(0.25)
q3 = payments["amount"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
payments["outlier_flag"] = payments["amount"] > upper
print("q1:", q1)
print("q3:", q3)
print("upper fence:", upper)
print("flagged rows:", int(payments["outlier_flag"].sum()))
print(payments[payments["outlier_flag"]].to_string(index=False))

q1: 1025.0
q3: 1550.0
upper fence: 2337.5
flagged rows: 1
 amount  outlier_flag
 180000          True


### Case C — extreme payment

The IQR rule creates a review list. It does not prove that the large amount is wrong or justify automatic deletion.

In [17]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(pd.DataFrame({"channel": ["branch", "web"]}))
new_rows = encoder.transform(pd.DataFrame({"channel": ["mobile", "web"]}))
print("feature names:", encoder.get_feature_names_out().tolist())
print("mobile:", new_rows[0].tolist())
print("web:", new_rows[1].tolist())

feature names: ['channel_branch', 'channel_web']
mobile: [0.0, 0.0]
web: [0.0, 1.0]


### Case D — unseen category

The unseen mobile channel receives zeros in the existing one-hot columns, while the monitoring process should record that a new category appeared.

In [18]:
train_demo = np.array([10.0, 20.0, 30.0])
test_demo = np.array([1000.0])
wrong_mean = np.concatenate([train_demo, test_demo]).mean()
right_mean = train_demo.mean()
print("full-sample mean:", wrong_mean)
print("training-only mean:", right_mean)
print("fit_transform belongs on training rows; transform belongs on test rows")

full-sample mean: 265.0
training-only mean: 20.0
fit_transform belongs on training rows; transform belongs on test rows


### Case E — leakage

The full-sample mean is 265, while the training-only mean is 20. The test observation has changed the representation in the wrong workflow.